# Лабораторная работа №2 — Car Price Prediction

**Цель:** обучить и сравнить модели регрессии стоимости автомобиля; сохранить результаты экспериментов в MLflow 2.16.0 (SQLite), зарегистрировать итоговую модель и назначить ей `Production`.

Используются **ровно 8 исходных столбцов ЛР1**. Новые инженерные признаки появляются **только внутри пайплайна моделирования (п. 10)** и не записываются в исходный датасет.

Выполняются обязательные пункты 1–10, 12, 15, заключительные 16–19. **Не выполняются пункты 11, 13, 14 и 16(*) с CatBoost.**

> Перед выполнением: запусти `.mlflow/start_mlflow.sh` в отдельном терминале; убедись, что в `data/clean_dataset.pkl` находится очищенная таблица ЛР1. Работай из `~/my_proj` с прежним виртуальным окружением.

## 1–2. Импорт библиотек и пути проекта

Проверяем, что виртуальное окружение активно, а исходные данные предыдущей лабораторной доступны. Не изменяем их.

In [ ]:
from pathlib import Path
import json
import shutil
import warnings

import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from IPython.display import display, Markdown

SOURCE_COLUMNS = (
    "name", "year", "selling_price", "km_driven",
    "fuel", "seller_type", "transmission", "owner"
)
TARGET = "selling_price"
NUMERIC_COLS = ["year", "km_driven"]
CATEGORICAL_COLS = ["name", "fuel", "seller_type", "transmission", "owner"]
INPUT_COLS = NUMERIC_COLS + CATEGORICAL_COLS

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (cwd, cwd.parent) if (p / "data" / "clean_dataset.pkl").is_file()),
    None
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Не найден data/clean_dataset.pkl. Выполни сохранение данных в ЛР1.")
RESEARCH_DIR = PROJECT_ROOT / "research"
RESEARCH_DIR.mkdir(exist_ok=True)
CLEAN_PATH = PROJECT_ROOT / "data" / "clean_dataset.pkl"
REQUIREMENTS = PROJECT_ROOT / "requirements.txt"
assert REQUIREMENTS.is_file(), "В корне нужен requirements.txt"

print("Проект:", PROJECT_ROOT)
print("Датасет:", CLEAN_PATH)
print("MLflow", mlflow.__version__, "| numpy", np.__version__)

## 3–4. Загрузка и разделение на train/test (75/25)

Числовые столбцы — `year`, `km_driven`. Категориальные — `name`, `fuel`, `seller_type`, `transmission`, `owner`. Цель — `selling_price`. Тестовая выборка до финального переобучения остаётся отдельной.

In [ ]:
# Загружаем только локальный файл, созданный в ЛР1 (доверенный Pickle).
df = pd.read_pickle(CLEAN_PATH)
assert set(SOURCE_COLUMNS) == set(df.columns), (
    f"Неожиданные столбцы: {df.columns.tolist()}"
)
df = df.loc[:, list(SOURCE_COLUMNS)].copy()

# Нормализуем типы, не добавляя новых столбцов.
for col in CATEGORICAL_COLS:
    df[col] = df[col].astype("string").str.strip()
for col in NUMERIC_COLS + [TARGET]:
    df[col] = pd.to_numeric(df[col], errors="raise")

assert df.notna().all().all(), "В очищенной таблице ЛР1 остались пропуски"
assert (df[TARGET] > 0).all(), "Цены должны быть положительными для MAPE"
assert len(df) >= 300, "По методичке нужно не менее 300 объектов"

X = df[INPUT_COLS].copy()
# Для MLflow и sklearn используем обычные строки вместо pandas.StringDtype.
for col in CATEGORICAL_COLS:
    X[col] = X[col].astype(str)
y = df[TARGET].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

print("Вся выборка:", X.shape)
print("Обучение:", X_train.shape, "| тест:", X_test.shape)
print("Числовые:", NUMERIC_COLS)
print("Категориальные:", CATEGORICAL_COLS)
print("Цель:", TARGET)
display(X_train.head())

## 5–6. Baseline-модель: `StandardScaler` + `OrdinalEncoder` + `RandomForestRegressor`

Все операции предобработки обучаются только на `X_train`. Значения `unknown_value=-1` позволяют обрабатывать ранее не встречавшиеся категории в тестовой выборке. Считаем **MAE, MAPE, MSE**.

In [ ]:
def baseline_preprocessor():
    return ColumnTransformer(
        transformers=[
            ("numeric", StandardScaler(), NUMERIC_COLS),
            (
                "categorical",
                OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
                CATEGORICAL_COLS,
            ),
        ],
        remainder="drop",
        verbose_feature_names_out=True,
    )


def engineered_preprocessor():
    # Для двух числовых признаков: degree=2, включая взаимодействие year*km_driven.
    # Масштабирование выполняем после PolynomialFeatures.
    poly_numeric = Pipeline([
        ("polynomial", PolynomialFeatures(degree=2, include_bias=False)),
        ("scaler", StandardScaler()),
    ])
    return ColumnTransformer(
        transformers=[
            ("numeric_poly", poly_numeric, NUMERIC_COLS),
            (
                "categorical",
                OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
                CATEGORICAL_COLS,
            ),
        ],
        remainder="drop",
        verbose_feature_names_out=True,
    )


def random_forest(n_estimators=80):
    return RandomForestRegressor(
        n_estimators=n_estimators,
        random_state=42,
        n_jobs=1,
    )


def evaluate(model, test_data=X_test, test_target=y_test):
    predictions = model.predict(test_data)
    return {
        "mae": float(mean_absolute_error(test_target, predictions)),
        "mape": float(mean_absolute_percentage_error(test_target, predictions)),
        "mse": float(mean_squared_error(test_target, predictions)),
    }


baseline_model = Pipeline([
    ("transform", baseline_preprocessor()),
    ("regressor", random_forest()),
])
baseline_model.fit(X_train, y_train)
baseline_metrics = evaluate(baseline_model)
print("Baseline:", baseline_metrics)

# Все результаты храним для сравнения.
results = {"baseline": baseline_metrics}
models = {"baseline": baseline_model}
run_ids = {}

## 7–8. Проверка MLflow (сервер должен быть запущен)

Перед этой ячейкой в **отдельном терминале** выполни:

```bash
cd ~/my_proj
source .venv_my_proj/bin/activate
cd .mlflow
bash start_mlflow.sh
```

Затем проверь <http://127.0.0.1:5000/>. Сервер хранит информацию в `.mlflow/mlruns.db`, артефакты — в `.mlflow/mlartifacts/`.

In [ ]:
TRACKING_URI = "http://127.0.0.1:5000"
EXPERIMENT_NAME = "car_price_prediction_lab2"
MODEL_NAME = "car_price_random_forest"

mlflow.set_tracking_uri(TRACKING_URI)

try:
    mlflow.set_experiment(EXPERIMENT_NAME)
except Exception as exc:
    raise RuntimeError(
        "Не удаётся подключиться к MLflow. "
        "Запусти .mlflow/start_mlflow.sh в отдельном терминале и открой "
        "http://127.0.0.1:5000/"
    ) from exc

client = MlflowClient()
print("MLflow подключён:", TRACKING_URI)


def log_experiment(run_name, trained_model, metrics, extra_params=None,
                   additional_artifacts=None):
    """Сохраняем модель, метрики, сигнатуру, пример входа, requirements.txt."""
    with mlflow.start_run(run_name=run_name) as run:
        mlflow.set_tag("laboratory", "lab2")
        mlflow.log_param("algorithm", "RandomForestRegressor")
        mlflow.log_param("input_features", ", ".join(INPUT_COLS))
        mlflow.log_param("train_rows", len(X_train))
        mlflow.log_param("test_rows", len(X_test))
        for param_name, param_value in (extra_params or {}).items():
            mlflow.log_param(param_name, str(param_value))
        for metric_name, metric_value in metrics.items():
            mlflow.log_metric(metric_name, metric_value)

        example = X_train.head(5).copy()
        signature = infer_signature(example, trained_model.predict(example))
        mlflow.sklearn.log_model(
            sk_model=trained_model,
            artifact_path="model",
            input_example=example,
            signature=signature,
        )
        mlflow.log_artifact(str(REQUIREMENTS))
        for path in (additional_artifacts or []):
            mlflow.log_artifact(str(path))
        return run.info.run_id


baseline_run_id = log_experiment(
    run_name="baseline_random_forest",
    trained_model=baseline_model,
    metrics=baseline_metrics,
    extra_params={"feature_engineering": False, "selection": False},
)
run_ids["baseline"] = baseline_run_id
print("Baseline run_id:", baseline_run_id)

## 9. Регистрация baseline-модели (версия 1)

Регистрируем модель как `car_price_random_forest`. При повторном запуске ноутбука номера версий увеличатся — это нормальное поведение MLflow.

In [ ]:
baseline_version = mlflow.register_model(
    model_uri=f"runs:/{baseline_run_id}/model",
    name=MODEL_NAME,
)
print("Модель:", MODEL_NAME)
print("Baseline, версия:", baseline_version.version)

## 10. Инженерия признаков средствами scikit-learn

Применяем `PolynomialFeatures(degree=2)` **только к двум числовым входным признакам**. Создаваемые преобразованные признаки существуют лишь внутри `Pipeline`; исходный `clean_dataset.pkl` не изменяется. Сохраняем названия преобразованных признаков.

In [ ]:
engineered_model = Pipeline([
    ("transform", engineered_preprocessor()),
    ("regressor", random_forest()),
])
engineered_model.fit(X_train, y_train)
engineered_metrics = evaluate(engineered_model)

engineered_columns = engineered_model.named_steps["transform"].get_feature_names_out().tolist()
columns_file = RESEARCH_DIR / "engineered_feature_names.txt"
columns_file.write_text("\n".join(engineered_columns) + "\n", encoding="utf-8")

print("Сгенерированные имена:", engineered_columns)
print("Количество:", len(engineered_columns))
print("Метрики:", engineered_metrics)

engineered_run_id = log_experiment(
    "feature_engineering_polynomial",
    engineered_model,
    engineered_metrics,
    extra_params={"polynomial_degree": 2, "features_count": len(engineered_columns)},
    additional_artifacts=[columns_file],
)
models["feature_engineering"] = engineered_model
results["feature_engineering"] = engineered_metrics
run_ids["feature_engineering"] = engineered_run_id
print("Feature engineering run_id:", engineered_run_id)

## 12. Отбор 50% преобразованных признаков методом SFS (sklearn)

`SequentialFeatureSelector` отбирает **5 из 10** преобразованных признаков. Для скорости используем небольшой лес внутри процедуры отбора; финальный RandomForest после отбора обучается отдельно. Индексы относятся к выходу `ColumnTransformer`, а не к восьми исходным столбцам.

In [ ]:
# Исследуемый набор состоит из 10 преобразованных признаков (5 polynomial + 5 категориальных).
selector_estimator = RandomForestRegressor(
    n_estimators=20, max_depth=10, random_state=42, n_jobs=1
)
selected_model = Pipeline([
    ("transform", engineered_preprocessor()),
    (
        "feature_selection",
        SequentialFeatureSelector(
            selector_estimator,
            n_features_to_select=5,
            direction="forward",
            scoring="neg_mean_absolute_error",
            cv=2,
            n_jobs=2,
        ),
    ),
    ("regressor", random_forest()),
])
selected_model.fit(X_train, y_train)
selected_metrics = evaluate(selected_model)

all_engineered_names = selected_model.named_steps["transform"].get_feature_names_out()
selected_indices = selected_model.named_steps["feature_selection"].get_support(indices=True)
selected_names = all_engineered_names[selected_indices].tolist()

selected_names_path = RESEARCH_DIR / "selected_feature_names.txt"
selected_indices_path = RESEARCH_DIR / "selected_feature_indices.txt"
selected_names_path.write_text("\n".join(selected_names) + "\n", encoding="utf-8")
selected_indices_path.write_text("\n".join(map(str, selected_indices.tolist())) + "\n", encoding="utf-8")

print("Выбранные признаки:", selected_names)
print("Индексы:", selected_indices.tolist())
print("Метрики:", selected_metrics)

selected_run_id = log_experiment(
    "feature_selection_sfs",
    selected_model,
    selected_metrics,
    extra_params={"selector": "SequentialFeatureSelector", "selected_count": len(selected_names)},
    additional_artifacts=[selected_names_path, selected_indices_path],
)
models["feature_selection"] = selected_model
results["feature_selection"] = selected_metrics
run_ids["feature_selection"] = selected_run_id
print("Feature selection run_id:", selected_run_id)

## 15. GridSearchCV: подбираем количество деревьев, глубину и max_features

Выполняем **8 комбинаций × 3 фолда = 24 обучения**. Минимизируем `MAE`, поэтому `scoring='neg_mean_absolute_error'`. Основой служит лучший из предыдущих кандидатов по тестовой MAE; если им стал вариант SFS, переносим отобранные по train индексы в фиксированный селектор, чтобы не повторять дорогой SFS на каждой комбинации GridSearch.

In [ ]:
previous_best = min(results, key=lambda key: results[key]["mae"])
print("Лучшая модель до GridSearchCV:", previous_best)

if previous_best == "baseline":
    base_for_grid = Pipeline([
        ("transform", baseline_preprocessor()),
        ("regressor", random_forest()),
    ])
elif previous_best == "feature_engineering":
    base_for_grid = Pipeline([
        ("transform", engineered_preprocessor()),
        ("regressor", random_forest()),
    ])
else:
    # Воспроизводим выбранные на train индексы штатным ColumnTransformer,
    # без пользовательских классов / lambda (важно для развёртывания MLflow).
    fixed_selection = ColumnTransformer(
        [("keep", "passthrough", selected_indices.tolist())],
        remainder="drop",
    )
    base_for_grid = Pipeline([
        ("transform", engineered_preprocessor()),
        ("feature_selection", fixed_selection),
        ("regressor", random_forest()),
    ])

param_grid = {
    "regressor__n_estimators": [40, 80],
    "regressor__max_depth": [10, None],
    "regressor__max_features": [0.5, 1.0],
}

grid = GridSearchCV(
    estimator=base_for_grid,
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",
    cv=3,
    n_jobs=2,
    refit=True,
    verbose=1,
)
grid.fit(X_train, y_train)

grid_model = grid.best_estimator_
grid_metrics = evaluate(grid_model)

print("Лучшие параметры:", grid.best_params_)
print("Лучшее CV MAE:", -grid.best_score_)
print("Метрики на отложенном тесте:", grid_metrics)

results["grid_search"] = grid_metrics
models["grid_search"] = grid_model

grid_run_id = log_experiment(
    "gridsearch_random_forest",
    grid_model,
    grid_metrics,
    extra_params={
        "grid_based_on": previous_best,
        "cv_mae": -grid.best_score_,
        **grid.best_params_,
    },
    additional_artifacts=(
        [selected_names_path, selected_indices_path]
        if previous_best == "feature_selection" else []
    ),
)
run_ids["grid_search"] = grid_run_id

grid_version = mlflow.register_model(
    model_uri=f"runs:/{grid_run_id}/model",
    name=MODEL_NAME,
)
print("GridSearch версия:", grid_version.version)
print("GridSearch run_id:", grid_run_id)

## Заключительный пункт 16. Сравнение экспериментов и обучение Production-модели на всей выборке

По методичке выбираем минимальную **MAE** на отложенной тестовой выборке. В реальном ML-проекте сравнение и выбор архитектуры лучше проводить по отдельной валидации или CV, сохраняя тест нетронутым до финальной оценки — здесь следуем лабораторному заданию.

После выбора клонируем пайплайн и обучаем **на всех очищенных объектах**. Для Production-run не пересчитываем тестовые метрики. Сохраняем новую версию модели, сигнатуру, пример входа, requirements.txt и список признаков; назначаем стадию и тег `Production`.

In [ ]:
# Сравниваем результаты по MAE.
comparison = pd.DataFrame(results).T[["mae", "mape", "mse"]]
comparison = comparison.sort_values("mae")
comparison.index.name = "run_name"
display(comparison)

best_name = comparison.index[0]
print("Выбрана модель:", best_name)
print("Метрики выбранной модели на тесте:", results[best_name])

# Повторное обучение на всей выборке.
production_model = clone(models[best_name])
production_model.fit(X, y)

final_artifacts = []
if best_name in ("feature_engineering", "feature_selection", "grid_search"):
    final_artifacts.append(columns_file)
if best_name == "feature_selection" or (
    best_name == "grid_search" and previous_best == "feature_selection"
):
    final_artifacts += [selected_names_path, selected_indices_path]

# Для Production логируем модель, сигнатуру, пример и требования.
# Метрики в этой версии НЕ пересчитываем (согласно методичке).
with mlflow.start_run(run_name="production_full_dataset") as run:
    mlflow.set_tag("laboratory", "lab2")
    mlflow.set_tag("purpose", "Production")
    mlflow.log_param("selected_model", best_name)
    mlflow.log_param("full_dataset_rows", len(X))
    mlflow.log_param("input_features", ", ".join(INPUT_COLS))
    example = X.head(5).copy()
    signature = infer_signature(example, production_model.predict(example))
    mlflow.sklearn.log_model(
        sk_model=production_model,
        artifact_path="model",
        signature=signature,
        input_example=example,
    )
    mlflow.log_artifact(str(REQUIREMENTS))
    for artifact in final_artifacts:
        mlflow.log_artifact(str(artifact))
    production_run_id = run.info.run_id

production_version = mlflow.register_model(
    model_uri=f"runs:/{production_run_id}/model",
    name=MODEL_NAME,
)

client.set_model_version_tag(
    name=MODEL_NAME,
    version=production_version.version,
    key="deployment_stage",
    value="Production",
)
# MLflow 2.16 ещё поддерживает stages; выставляем именно Production.
client.transition_model_version_stage(
    name=MODEL_NAME,
    version=production_version.version,
    stage="Production",
    archive_existing_versions=True,
)

# По методичке копируем оригинальный MLmodel (метаданные формата MLflow)
# из артефактов именно Production-run; модельные бинарники не копируем в Git.
local_mlmodel_path = client.download_artifacts(
    production_run_id, "model/MLmodel"
)
shutil.copyfile(local_mlmodel_path, RESEARCH_DIR / "MLmodel")

print("PRODUCTION MODEL:", MODEL_NAME)
print("Version:", production_version.version)
print("run_id:", production_run_id)
print("MLmodel скопирован в:", RESEARCH_DIR / "MLmodel")

## 17–18. Артефакты, скриншоты, README и Git

**Пользовательские скриншоты выполняются в браузере после реального запуска:**

1. `research/mlflow_metrics.png` — эксперимент `car_price_prediction_lab2`, таблица Runs / сравнение MAE, MAPE, MSE.
2. `research/mlflow_registry.png` — Registered Models → `car_price_random_forest`, версии и `Production`.

**Файл `research/MLmodel` создаётся предыдущей ячейкой** как копия из артефактов настоящей Production-модели, не заглушка.

Следующая ячейка обновит README, добавив название лучшей модели, параметры, значения MAE/MAPE/MSE, номер версии и `run_id`. Далее обнови `.gitignore` по `research/gitignore_lab2.txt`, проверь `requirements.txt` и отправь файлы на GitHub. `.mlflow/mlruns.db` и `.mlflow/mlartifacts` туда не отправляй.

In [ ]:
# Формируем документ со статистикой фактических запусков.
metric_lines = [
    "| Модель | MAE | MAPE | MSE | run_id |",
    "|---|---:|---:|---:|---|",
]
for name, row in comparison.iterrows():
    metric_lines.append(
        f"| {name} | {row['mae']:.3f} | {row['mape']:.5f} | "
        f"{row['mse']:.3f} | `{run_ids[name]}` |"
    )

params = production_model.named_steps["regressor"].get_params()

lab2_report = f"""## Лабораторная работа №2 — исследование моделей

Использована очищенная таблица `data/clean_dataset.pkl` из ЛР1. Входные
признаки: `{', '.join(INPUT_COLS)}`. Цель: `{TARGET}`.

- Обучающая/тестовая выборки: 75% / 25%, `random_state=42`.
- Модель: `RandomForestRegressor`.
- Числовые признаки: `StandardScaler`.
- Категориальные признаки: `OrdinalEncoder`.
- Инженерия признаков: `PolynomialFeatures(degree=2)`.
- Отбор 5 из 10 полученных признаков: `SequentialFeatureSelector`.
- Подбор гиперпараметров: `GridSearchCV`, 8 комбинаций, 3 фолда, оптимизация MAE.

### Метрики на отложенной выборке

{chr(10).join(metric_lines)}

### Выбранная модель и Production

- Лучший вариант по тестовой MAE: **{best_name}**.
- Параметры RandomForest: `n_estimators={params['n_estimators']}`,
  `max_depth={params['max_depth']}`, `max_features={params['max_features']}`.
- На всей очищенной выборке выполнено повторное обучение.
- Реестр MLflow: `{MODEL_NAME}`.
- Версия модели: **{production_version.version}**.
- Production `run_id`: `{production_run_id}`.
- MLflow: <http://127.0.0.1:5000/> (только при работающем локальном сервере).
- Метаданные: `research/MLmodel`.

Скриншоты после выполнения: `research/mlflow_metrics.png`,
`research/mlflow_registry.png`.

Новые признаки создаются **только внутри пайплайна**, исходные восемь
столбцов очищенного датасета не изменяются.

Для запуска MLflow: активировать `.venv_my_proj`, затем выполнить
`cd .mlflow && bash start_mlflow.sh`.
"""

report_path = RESEARCH_DIR / "lab2_results.md"
report_path.write_text(lab2_report, encoding="utf-8")

readme_path = PROJECT_ROOT / "README.md"
readme = readme_path.read_text(encoding="utf-8")
start_mark = "<!-- LAB2_RESULTS_START -->"
end_mark = "<!-- LAB2_RESULTS_END -->"
section = start_mark + "\n" + lab2_report + "\n" + end_mark
if start_mark in readme and end_mark in readme:
    start_idx = readme.index(start_mark)
    end_idx = readme.index(end_mark) + len(end_mark)
    readme = readme[:start_idx] + section + readme[end_idx:]
else:
    readme = readme.rstrip() + "\n\n" + section + "\n"
readme_path.write_text(readme, encoding="utf-8")

print("README обновлён:", readme_path)
print("Отчёт создан:", report_path)
print("run_id Production:", production_run_id)
display(Markdown(lab2_report))

## 19. Передача работы

После создания двух настоящих скриншотов и проверки результатов выполни из корня проекта:

```bash
cat research/gitignore_lab2.txt >> .gitignore
python research/update_requirements.py
# Предварительно проверь diff и список файлов
git add README.md requirements.txt .gitignore .mlflow/start_mlflow.sh research/research.ipynb   research/MLmodel research/lab2_results.md research/selected_feature_names.txt   research/selected_feature_indices.txt research/engineered_feature_names.txt   research/mlflow_metrics.png research/mlflow_registry.png

git diff --cached --name-only
git commit -m "Complete lab 2: RandomForest, MLflow and feature selection"
git push origin main
```

До `git add` убедись, что указанные файлы действительно созданы. Для сохранения данных ЛР2 отдельно от GitHub сделай резервную копию **всей директории** `.mlflow` (она должна содержать SQLite и `mlartifacts`). При восстановлении желательно оставить тот же путь проекта либо перенастроить URI хранилища артефактов.